In [1]:
import logging
import psycopg2
from psycopg2 import IntegrityError, OperationalError
from psycopg2.extras import RealDictCursor

logging.basicConfig(
    filename="etl.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("banking_etl")
print("Logging configured -> etl.log")

Logging configured -> etl.log


In [2]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.conn = psycopg2.connect(
            host=host, port=port, dbname=dbname, user=user, password=password
        )
        logger.info("Connected to database '%s'", dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")

## Task 2 — Harden the Connection


In [7]:
import time
import functools

def with_retry(max_attempts=3, delay=1):
    """Retries a DB operation on OperationalError/InterfaceError, rolling back each time."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(db, *args, **kwargs):
            attempt = 1
            while True:
                try:
                    return func(db, *args, **kwargs)
                except (OperationalError, psycopg2.InterfaceError) as e:
                    logger.warning(
                        "Attempt %d/%d failed for %s: %s",
                        attempt, max_attempts, func.__name__, e
                    )
                    try:
                        db.rollback()
                    except Exception:
                        pass
                    if attempt >= max_attempts:
                        logger.error(
                            "Max retries (%d) reached for %s. Giving up.",
                            max_attempts, func.__name__
                        )
                        raise
                    attempt += 1
                    time.sleep(delay)
        return wrapper
    return decorator

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
db = DBConnection(
    host="localhost",
    port=5432,
    dbname="db_bank",
    user=os.getenv("DB_USER", "your_user"),
    password=os.getenv("DB_PASSWORD", "your_password"),
)

In [9]:
@with_retry(max_attempts=3, delay=1)
def create_schema(db):
    ddl = """
    CREATE TABLE IF NOT EXISTS customers (
        customer_id SERIAL PRIMARY KEY,
        name        TEXT NOT NULL,
        email       TEXT UNIQUE
    );

    CREATE TABLE IF NOT EXISTS accounts (
        account_id  SERIAL PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(customer_id),
        balance     NUMERIC(12, 2) NOT NULL DEFAULT 0,
        status      TEXT NOT NULL DEFAULT 'open'
    );

    CREATE TABLE IF NOT EXISTS transactions (
        transaction_id SERIAL PRIMARY KEY,
        customer_id    INTEGER REFERENCES customers(customer_id),
        amount         NUMERIC(12, 2) NOT NULL,
        transaction_date DATE NOT NULL DEFAULT CURRENT_DATE
    );

    CREATE TABLE IF NOT EXISTS account_summary (
        customer_id       INTEGER PRIMARY KEY REFERENCES customers(customer_id),
        name              TEXT NOT NULL,
        total_transactions INTEGER NOT NULL,
        total_amount      NUMERIC(12, 2) NOT NULL,
        account_category  TEXT NOT NULL,
        high_value_flag   BOOLEAN NOT NULL,
        total_loan_exposure NUMERIC,
        active_loan_count    INTEGER,
        updated_at        TIMESTAMP NOT NULL DEFAULT NOW()
    );

    CREATE TABLE IF NOT EXISTS loans (
        loan_id       SERIAL PRIMARY KEY,
        customer_id   INTEGER REFERENCES customers(customer_id),
        principal     NUMERIC(12, 2) NOT NULL,
        interest_rate NUMERIC(5, 2) NOT NULL,
        status        TEXT NOT NULL CHECK (status IN ('active','paid_off','defaulted')),
        start_date    DATE NOT NULL DEFAULT CURRENT_DATE
    );
    """
    try:
        with db.cursor() as cur:
            cur.execute(ddl)
            cur.execute("""
                ALTER TABLE account_summary
                ADD COLUMN IF NOT EXISTS total_loan_exposure NUMERIC,
                ADD COLUMN IF NOT EXISTS active_loan_count INTEGER;
            """)
        db.commit()
        logger.info("Schema created/verified")
        print("Schema created/verified")
    except OperationalError as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise

create_schema(db)

Schema created/verified


In [12]:
@with_retry(max_attempts=3, delay=1)
def seed_sample_data(db):
    customers = [("Alice Shrestha", "alice@example.com"),
                 ("Bibek Rai", "bibek@example.com"),
                 ("Sita Gurung", "sita@example.com")]
    accounts = [(1, 1500.00, "open"), (2, 30200.00, "open"), (3, 9050.00, "open")]
    transactions = [(1, 500.00), (1, 15000.00), (3, 9000.00),
                     (3, 50.00), (2, 30000.00)]
    loans = [
        (1, 5000.00, 4.5, "active"),
        (1, 2000.00, 6.0, "paid_off"),
        (2, 15000.00, 5.25, "active"),
        (2, 8000.00, 7.0, "defaulted"),
        (3, 3000.00, 3.9, "active"),
        (3, 1000.00, 5.0, "paid_off"),
    ]
    try:
        with db.cursor() as cur:
            cur.executemany(
                "INSERT INTO customers (name, email) VALUES (%s, %s) "
                "ON CONFLICT (email) DO NOTHING",
                customers,
            )
            cur.executemany(
                "INSERT INTO accounts (customer_id, balance, status) VALUES (%s, %s, %s)",
                accounts,
            )
            cur.executemany(
                "INSERT INTO transactions (customer_id, amount) VALUES (%s, %s)",
                transactions,
            )
            cur.executemany(
                "INSERT INTO loans (customer_id, principal, interest_rate, status) "
                "VALUES (%s, %s, %s, %s)",
                loans,
            )
        db.commit()
        logger.info("Sample data seeded")
        print("Sample data seeded")
    except IntegrityError as e:
        db.rollback()
        logger.error("Seeding failed: %s", e)

seed_sample_data(db)

Sample data seeded


In [11]:
# Task 2 demonstration: simulate a dropped connection
print("Simulating a dropped connection before seeding...")
db.close()

try:
    seed_sample_data(db)  
except Exception as e:
    print(f"Final failure after retries, as expected: {e}")


db = DBConnection(
    host="localhost",
    port=5432,
    dbname="db_bank",
    user=os.getenv("DB_USER", "your_user"),
    password=os.getenv("DB_PASSWORD", "your_password"),
)

Simulating a dropped connection before seeding...
Final failure after retries, as expected: connection already closed


## Task 3: Extend the Transform Step


In [13]:
def extract(db):
    query = """
        SELECT c.customer_id, c.name, t.amount, t.transaction_date
        FROM customers c
        JOIN transactions t ON c.customer_id = t.customer_id
    """
    with db.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
    logger.info("Extracted %d transaction rows", len(rows))
    return rows

raw_rows = extract(db)
raw_rows

[RealDictRow([('customer_id', 1),
              ('name', 'Alice Shrestha'),
              ('amount', Decimal('500.00')),
              ('transaction_date', datetime.date(2026, 9, 13))]),
 RealDictRow([('customer_id', 1),
              ('name', 'Alice Shrestha'),
              ('amount', Decimal('15000.00')),
              ('transaction_date', datetime.date(2026, 9, 13))]),
 RealDictRow([('customer_id', 3),
              ('name', 'Sita Gurung'),
              ('amount', Decimal('9000.00')),
              ('transaction_date', datetime.date(2026, 9, 13))]),
 RealDictRow([('customer_id', 3),
              ('name', 'Sita Gurung'),
              ('amount', Decimal('50.00')),
              ('transaction_date', datetime.date(2026, 9, 13))]),
 RealDictRow([('customer_id', 2),
              ('name', 'Bibek Rai'),
              ('amount', Decimal('30000.00')),
              ('transaction_date', datetime.date(2026, 9, 13))])]

In [14]:
def extract_loans(db):
    query = "SELECT customer_id, principal, status FROM loans"
    with db.cursor() as cur:
        cur.execute(query)
        rows = cur.fetchall()
    logger.info("Extracted %d loan rows", len(rows))
    return rows

loan_rows = extract_loans(db)
loan_rows

[RealDictRow([('customer_id', 1),
              ('principal', Decimal('5000.00')),
              ('status', 'active')]),
 RealDictRow([('customer_id', 1),
              ('principal', Decimal('2000.00')),
              ('status', 'paid_off')]),
 RealDictRow([('customer_id', 2),
              ('principal', Decimal('15000.00')),
              ('status', 'active')]),
 RealDictRow([('customer_id', 2),
              ('principal', Decimal('8000.00')),
              ('status', 'defaulted')]),
 RealDictRow([('customer_id', 3),
              ('principal', Decimal('3000.00')),
              ('status', 'active')]),
 RealDictRow([('customer_id', 3),
              ('principal', Decimal('1000.00')),
              ('status', 'paid_off')])]

In [15]:
def categorize(total, has_defaulted_loan=False):
    if has_defaulted_loan:
        return "At Risk"
    if total >= 20000:
        return "Premium"
    elif total >= 5000:
        return "Standard"
    else:
        return "Basic"

In [16]:
def transform_loans(rows):
    per_customer = {}
    for row in rows:
        cid = row["customer_id"]
        entry = per_customer.setdefault(
            cid, {"total_loan_exposure": 0.0, "active_loan_count": 0, "has_defaulted": False}
        )
        if row["status"] == "active":
            entry["total_loan_exposure"] += float(row["principal"])
            entry["active_loan_count"] += 1
        if row["status"] == "defaulted":
            entry["has_defaulted"] = True

    logger.info("Transformed loan data for %d customers", len(per_customer))
    return per_customer

In [17]:
def transform(rows, loan_rows):
    HIGH_VALUE_THRESHOLD = 10000
    per_customer = {}

    for row in rows:
        cid = row["customer_id"]
        entry = per_customer.setdefault(
            cid, {"name": row["name"], "total": 0.0, "count": 0, "high_value": False}
        )
        entry["total"] += float(row["amount"])
        entry["count"] += 1
        if float(row["amount"]) >= HIGH_VALUE_THRESHOLD:
            entry["high_value"] = True

    loan_summary = transform_loans(loan_rows)

    summary_rows = []
    for cid, data in per_customer.items():
        loan_data = loan_summary.get(
            cid, {"total_loan_exposure": 0.0, "active_loan_count": 0, "has_defaulted": False}
        )
        summary_rows.append((
            cid,
            data["name"],
            data["count"],
            round(data["total"], 2),
            categorize(data["total"], loan_data["has_defaulted"]),
            data["high_value"],
            loan_data["total_loan_exposure"],
            loan_data["active_loan_count"],
        ))

    logger.info("Transformed data for %d customers", len(summary_rows))
    return summary_rows

summary_rows = transform(raw_rows, loan_rows)
summary_rows

[(1, 'Alice Shrestha', 2, 15500.0, 'Standard', True, 5000.0, 1),
 (3, 'Sita Gurung', 2, 9050.0, 'Standard', False, 3000.0, 1),
 (2, 'Bibek Rai', 1, 30000.0, 'At Risk', True, 15000.0, 1)]

## Task 4: Load & Verify

In [18]:
def load_summary(db, summary_rows):
    upsert_sql = """
        INSERT INTO account_summary
            (customer_id, name, total_transactions, total_amount,
             account_category, high_value_flag,
             total_loan_exposure, active_loan_count, updated_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, NOW())
        ON CONFLICT (customer_id) DO UPDATE SET
            name = EXCLUDED.name,
            total_transactions = EXCLUDED.total_transactions,
            total_amount = EXCLUDED.total_amount,
            account_category = EXCLUDED.account_category,
            high_value_flag = EXCLUDED.high_value_flag,
            total_loan_exposure = EXCLUDED.total_loan_exposure,
            active_loan_count = EXCLUDED.active_loan_count,
            updated_at = NOW();
    """
    try:
        with db.cursor() as cur:
            cur.executemany(upsert_sql, summary_rows)
        db.commit()
        logger.info("Loaded/updated %d summary rows", len(summary_rows))
        print(f"Loaded/updated {len(summary_rows)} summary rows")
    except (IntegrityError, OperationalError) as e:
        db.rollback()
        logger.error("Load failed: %s", e)
        raise

load_summary(db, summary_rows)

Loaded/updated 3 summary rows


In [19]:
def verify(db):
    cte_query = """
        WITH source_counts AS (
            SELECT COUNT(DISTINCT customer_id) AS n FROM transactions
        ),
        summary_counts AS (
            SELECT COUNT(*) AS n FROM account_summary
        )
        SELECT source_counts.n AS source_customers,
               summary_counts.n AS summary_customers
        FROM source_counts, summary_counts;
    """
    join_query = """
        SELECT s.customer_id, s.name, s.total_transactions,
               s.total_amount, s.account_category, s.high_value_flag,
               s.total_loan_exposure, s.active_loan_count
        FROM account_summary s
        JOIN customers c ON c.customer_id = s.customer_id
        ORDER BY s.total_amount DESC;
    """
    with db.cursor() as cur:
        cur.execute(cte_query)
        counts = cur.fetchone()
        cur.execute(join_query)
        results = cur.fetchall()

    logger.info("Verification: source=%s summary=%s",
                counts["source_customers"], counts["summary_customers"])

    print(f"Source customers: {counts['source_customers']} | Summary customers: {counts['summary_customers']}")
    if counts["source_customers"] != counts["summary_customers"]:
        print("MISMATCH: not every customer has a summary row!")
    else:
        print("OK: every customer in customers appears exactly once in account_summary.\n")

    print(f"{'ID':<4}{'Name':<16}{'Txns':<6}{'Total':<12}{'Category':<10}{'HighVal':<8}{'LoanExp':<10}{'ActiveLoans':<12}")
    for r in results:
        print(f"{r['customer_id']:<4}{r['name']:<16}{r['total_transactions']:<6}"
              f"{r['total_amount']:<12}{r['account_category']:<10}{str(r['high_value_flag']):<8}"
              f"{r['total_loan_exposure']:<10}{r['active_loan_count']:<12}")

    return counts, results

counts, results = verify(db)

Source customers: 3 | Summary customers: 3
OK: every customer in customers appears exactly once in account_summary.

ID  Name            Txns  Total       Category  HighVal LoanExp   ActiveLoans 
2   Bibek Rai       1     30000.00    At Risk   True    15000.0   1           
1   Alice Shrestha  2     15500.00    Standard  True    5000.0    1           
3   Sita Gurung     2     9050.00     Standard  False   3000.0    1           


## Task 5: Unit Test

In [24]:

!pytest test_transform.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\Bishal\AppData\Local\Programs\Python\Python312\python.exe
cachedir: .pytest_cache
rootdir: d:\CLASS ASSIGNMENTS\day-08-AI-Foundations
plugins: anyio-4.13.0
collecting ... collected 16 items

test_transform.py::test_categorize_premium_boundary PASSED               [  6%]
test_transform.py::test_categorize_just_below_premium PASSED             [ 12%]
test_transform.py::test_categorize_standard_boundary PASSED              [ 18%]
test_transform.py::test_categorize_below_standard_is_basic PASSED        [ 25%]
test_transform.py::test_categorize_defaulted_overrides_high_total PASSED [ 31%]
test_transform.py::test_categorize_defaulted_overrides_low_total PASSED  [ 37%]
test_transform.py::test_categorize_no_default_flag_behaves_normally PASSED [ 43%]
test_transform.py::test_transform_loans_zero_loans PASSED                [ 50%]
test_transform